In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from  sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [4]:
restuarant_data=pd.read_csv(r"C:\Users\haniy\python\Restaurant_Reviews.csv",sep="\t")
# restuarant_data=pd.read_csv(r"C:\Users\haniy\python\Restaurant_Reviews.csv",sep=",",on_bad_lines="skip)
restuarant_data.head(5)

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [5]:
restuarant_data.shape

(1000, 2)

In [6]:
restuarant_data.dtypes

Review    object
Liked      int64
dtype: object

In [7]:
restuarant_data.isnull().sum()

Review    0
Liked     0
dtype: int64

In [8]:
restuarant_data.describe()

,Liked
count,1000.00000
mean,0.50000
std,0.50025
min,0.00000
25%,0.00000
50%,0.50000
75%,1.00000
max,1.00000


In [9]:
#Class Distribution (Also checking balanced or imbalanced data)
restuarant_data["Liked"].value_counts()

Liked
1    500
0    500
Name: count, dtype: int64

In [10]:
sns.countplot(x="Liked",data=restuarant_data)
plt.title("Sentiment Analysis (0=Negative,1=Positive)")

Text(0.5, 1.0, 'Sentiment Analysis (0=Negative,1=Positive)')

## Process Text

In [11]:
#finding count of characters in review
restuarant_data["Char Count"]=restuarant_data["Review"].apply(len)

In [12]:
#finding largest and smallest reviews
restuarant_data["Word_count"]=restuarant_data["Review"].apply(lambda x:len(x.split()))
print("---------Highest Word Count------------")
print(restuarant_data.sort_values(by="Word_count",ascending=False).head(5)["Review"])
print("-----------Lowest Word Count-------------")
print(restuarant_data.sort_values(by="Word_count",ascending=True).head(5)["Review"])

---------Highest Word Count------------
623    a drive thru means you do not want to wait aro...
795    So good I am going to have to review this plac...
549    My boyfriend and I came here for the first tim...
210    If that bug never showed up I would have given...
123    The guys all had steaks, and our steak loving ...
Name: Review, dtype: object
-----------Lowest Word Count-------------
165            DELICIOUS!!
551         Nice ambiance.
824         Awful service.
664    Eclectic selection.
114           Good prices.
Name: Review, dtype: object


In [13]:
restuarant_data.head(5)

,Review,Liked,Char Count,Word_count
0,Wow... Loved this place.,1,24,4
1,Crust is not good.,0,18,4
2,Not tasty and the texture was just nasty.,0,41,8
3,Stopped by during the late May bank holiday of...,1,87,15
4,The selection on the menu was great and so wer...,1,59,12


In [14]:
#!pip install nltk

In [15]:
import re
import nltk
from nltk.corpus import stopwords

In [16]:
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('stopwords')
# nltk.download('wordnet')

In [17]:
restuarant_data["sent_count"]=restuarant_data["Review"].apply(lambda x:len(nltk.sent_tokenize(str(x))))
restuarant_data.head()

,Review,Liked,Char Count,Word_count,sent_count
0,Wow... Loved this place.,1,24,4,2
1,Crust is not good.,0,18,4,1
2,Not tasty and the texture was just nasty.,0,41,8,1
3,Stopped by during the late May bank holiday of...,1,87,15,1
4,The selection on the menu was great and so wer...,1,59,12,1


In [18]:
#Finding average Liked for word length
print(restuarant_data.groupby("Liked")["Word_count"].mean())

Liked
0    11.498
1    10.290
Name: Word_count, dtype: float64


In [19]:
#Finding average Liked for char length
print(restuarant_data.groupby("Liked")["Char Count"].mean())

Liked
0    60.75
1    55.88
Name: Char Count, dtype: float64


## Observation
- Negative reviews have higher word and character count than positive review

In [20]:
## Re Module

In [21]:
review=re.sub('[^a-zA-Z]',' ',restuarant_data['Review'][1]).lower()
review

'crust is not good '

In [22]:
#review=review.lower()

In [23]:
review=review.split()
review

['crust', 'is', 'not', 'good']

In [24]:
#removing stopwords
all_stopwords=stopwords.words("English")
all_stopwords.remove('not')

In [25]:
review=[word for word in review if word not in set(all_stopwords)]
review

['crust', 'not', 'good']

## Stemming

In [26]:
from nltk.stem.porter import PorterStemmer

In [27]:
ps=PorterStemmer()

In [28]:
review=[ps.stem(word) for word in review]

In [29]:
review=" ".join(review)
review

'crust not good'

In [30]:
custom_stopwords = {'don', "don't", 'ain', 'aren', "aren't", 'couldn', "couldn't",
                    'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't",
                    'haven', "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't",
                    'needn', "needn't", 'shan', "shan't", 'no', 'nor', 'not', 'shouldn', "shouldn't",
                    'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"}
corpus=[]
ps=PorterStemmer()
stop_words=set(stopwords.words("english"))-custom_stopwords
for i in range(len(restuarant_data)):
    review=re.sub('[^a-zA-Z]',' ',restuarant_data["Review"][i])
    review=review.lower()
    review=review.split()
    review=[ps.stem(word) for word in review if word not in stop_words]
    review=" ".join(review)
    corpus.append(review)

In [31]:
restuarant_data["processed_text"]=corpus
restuarant_data.head()

,Review,Liked,Char Count,Word_count,sent_count,processed_text
0,Wow... Loved this place.,1,24,4,2,wow love place
1,Crust is not good.,0,18,4,1,crust not good
2,Not tasty and the texture was just nasty.,0,41,8,1,not tasti textur nasti
3,Stopped by during the late May bank holiday of...,1,87,15,1,stop late may bank holiday rick steve recommen...
4,The selection on the menu was great and so wer...,1,59,12,1,select menu great price


In [32]:
# pip install wordcloud

In [33]:
from wordcloud import WordCloud

In [34]:
wc=WordCloud(width=500,height=500,min_font_size=6,background_color="white")

In [35]:
pos=wc.generate(restuarant_data[restuarant_data["Liked"]==1]["processed_text"].str.cat(sep=" "))

In [36]:
plt.imshow(pos)

In [37]:
#the larger size the higher ferequent appearence of the word

In [38]:
nos=wc.generate(restuarant_data[restuarant_data["Liked"]==0]["processed_text"].str.cat(sep=" "))
plt.imshow(nos)

In [39]:
#TfidfVectorizer => it is used to convert the text in the review to number for machine learning process

In [40]:
vectorizer=TfidfVectorizer(max_features=500)

In [41]:
X=vectorizer.fit_transform(restuarant_data["Review"]).toarray() #input data
y=restuarant_data["Liked"] #output data

In [42]:
X.shape   #now the X variable will have 1000 rows with 500 features

(1000, 500)

In [43]:
#Applying classification algorithm

In [44]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [45]:
nbmodel=GaussianNB()
nbmodel.fit(X_train,y_train)
nb_predict=nbmodel.predict(X_test)

In [46]:
lmodel=LogisticRegression()
lmodel.fit(X_train,y_train)
l_predict=lmodel.predict(X_test)

In [47]:
accuracy_score(y_test,nb_predict)

0.7266666666666667

In [48]:
accuracy_score(y_test,l_predict)

0.8066666666666666

In [49]:
rfmodel=RandomForestClassifier()
rfmodel.fit(X_train,y_train)
rf_predict=rfmodel.predict(X_test)

In [50]:
accuracy_score(y_test,rf_predict)

0.7566666666666667

## Observation
Among the three models LogisticRegression have high accuracy of the model performance.

In [53]:
# pip install joblib

In [54]:
import joblib

In [64]:
joblib.dump(lmodel,'Restaurant_review_model.pkl') #  saving logidtic reg model 

['Restaurant_review_model.pkl']

In [65]:
joblib.dump(vectorizer,'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']